[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/46.%20The%20Competitive%20Port%20Pricing%20Problem/P46-Tier-8_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 46. The Competitive Port Pricing Problem

## Tier 8: The Value-Aligned and Ethical Framework

### Goal and scope

This notebook demonstrates a rule-based ethical assessment and a stochastic, iterative search for port prices. Candidate prices are evaluated against ethical constraints and stakeholder utility functions. The purpose is to make the trade-offs and audit process inspectable; this implementation is an illustrative model, not a complete legal or operational compliance system.

The example uses three ports and three shipping lines to describe the pricing context, and a three-element price vector for the ethical demonstration. Port capacities, the logit-model parameters, demand allocation among ports, strategic interaction among ports, and legal rules are not used in the search itself. The price vector is interpreted as one price per shipping line. Stakeholder utility functions and constraint thresholds are modeling choices, not empirically validated measures of welfare or fairness. The framework represents customers, shareholders, the local community, and the environment; employees and regulators are named stakeholder categories but do not have utility functions in this example.

The candidate-generation method is a random walk with a decreasing step size. It is a heuristic search: it does not prove that the selected prices are globally optimal. Because no random seed is set, repeated runs may produce different results.

### Inputs, outputs, and learning objectives

- **Inputs:** base prices; market context, including reference prices, price bounds, costs, and demands; and the framework's constraints and stakeholder functions.
- **Outputs:** a selected price vector, constraint assessments, stakeholder utility values, an explanation, and an audit record.
- **Learning objectives:** distinguish hard constraints from soft constraints, understand a weighted scoring rule, trace an iterative random search, and interpret audit and bias-monitoring results without treating them as proof of real-world fairness.

### Modeling assumptions

Prices are positive monetary amounts in a common unit. The three entries of each price vector use the same shipping-line order as the demand and cost arrays. The shortage scenario compares prices with pre-shortage reference prices. The fairness check limits the mean price-to-reference-price ratio; the equity check compares the mean price for the small-customer segment with the mean across all customers. The non-discrimination proxy uses the coefficient of variation of prices, and the sustainability check uses a supplied environmental-impact index. These proxies do not establish whether a price difference is justified in a specific real market.

### Method and mathematical model

Let \(p^{(t)} \in \mathbb{R}^n\) be the current candidate price vector at iteration \(t\), and let \(\ell\) and \(u\) be the elementwise lower and upper price bounds. A candidate is generated by perturbing each price and clipping it to those bounds:

$$
\tilde p^{(t)} = \operatorname{clip}\!\left(p^{(t)} + \epsilon^{(t)},\, \ell,\, u\right),
\qquad
\epsilon_i^{(t)} \sim \operatorname{Uniform}(-s_t,s_t),
\qquad
s_t = 10(0.95)^t.
$$

Here, \(n\) is the number of prices, \(i\) indexes a price, and \(s_t\) is the decreasing perturbation size. Clipping keeps each proposed price within the market bounds. A hard constraint determines whether a candidate is eligible; a soft constraint may be violated, but its violation lowers the ethical score.

For a candidate \(p\), the implementation computes stakeholder utilities \(U_j(p)\), already multiplied by stakeholder weights \(w_j\), and an ethical score \(E(p)\). It ranks eligible candidates using:

$$
S(p) = 0.6E(p) + 0.4\left(\frac{\sum_j U_j(p)}{10}\right).
$$

The constants 0.6, 0.4, and 10 are the scoring choices used by this example. The selected candidate is the one with the greatest score among the candidates generated during the fixed 100-iteration search. The search retains the best candidate found so far; it is not a proof of a global optimum. If no hard-feasible candidate is found, the routine raises an error rather than returning an unassessed decision.

For a soft constraint with violation degree \(v_k(p)\) and penalty weight \(a_k\), the implemented penalty is \(a_k v_k(p)\). A violated hard constraint incurs its configured penalty in the compliance assessment, but hard-infeasible candidates are excluded from the search. The ethical score is \(\max(0,1-\text{total penalty})\). The base case for the search is an empty incumbent; each hard-feasible candidate replaces it only if its score is greater. With \(T\) iterations, \(n\) prices, and \(K\) constraints, the search work is approximately \(O(T(n+K))\), apart from the cost of stakeholder utility functions.

The code follows this sequence: define the market and stakeholder records; evaluate constraints; calculate weighted utilities; generate and score candidates; record the decision; then inspect bias, simulation behavior, and reporting. A high score is a result of these specified proxies and weights, not a general ethical certification.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from enum import Enum
from typing import Callable, Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Port:
    """A port with a capacity and a cost for each shipping line."""

    id: int
    name: str
    capacity: float
    costs: List[float]


@dataclass
class ShippingLine:
    """A shipping line with demand and customer characteristics."""

    id: int
    name: str
    demand: float
    priority: int
    size_category: str
    price_sensitivity: float


@dataclass
class MarketParameters:
    """Market-wide price bounds and price sensitivity."""

    price_sensitivity: float
    min_price: float
    max_price: float


class EthicalPrinciple(Enum):
    """Principles represented by the example framework."""

    FAIRNESS = "fairness"
    TRANSPARENCY = "transparency"
    ACCOUNTABILITY = "accountability"
    SUSTAINABILITY = "sustainability"
    EQUITY = "equity"
    NON_DISCRIMINATION = "non_discrimination"


class StakeholderType(Enum):
    """Stakeholder categories that can be represented in the model."""

    CUSTOMERS = "customers"
    EMPLOYEES = "employees"
    SHAREHOLDERS = "shareholders"
    COMMUNITY = "community"
    ENVIRONMENT = "environment"
    REGULATORS = "regulators"


@dataclass
class EthicalConstraint:
    """A hard or soft test applied to a candidate price vector."""

    principle: EthicalPrinciple
    constraint_type: str
    description: str
    check_function: Callable
    penalty_weight: float = 1.0
    threshold: float = 0.0


@dataclass
class Stakeholder:
    """A stakeholder and a utility function for the modeled context."""

    stakeholder_type: StakeholderType
    name: str
    interests: List[str]
    utility_function: Callable
    weight: float = 1.0


@dataclass
class EthicalDecision:
    """A pricing decision and its recorded assessment."""

    timestamp: float
    decision_maker: str
    prices: np.ndarray
    ethical_score: float
    stakeholder_utilities: Dict[str, float]
    constraint_compliance: Dict[str, bool]
    rationale: str
    audit_trail: List[str]
    approval_required: bool = False
    approved_by: Optional[str] = None

In [ ]:
class EthicalFramework:
    """Evaluate ethical constraints and search for high-scoring prices."""

    def __init__(self):
        self.ethical_principles = list(EthicalPrinciple)
        self.constraints = []
        self.stakeholders = []
        self.audit_log = []
        self.compliance_history = []
        self.bias_detector = BiasDetector()
        self.transparency_engine = TransparencyEngine()
        self._initialize_default_constraints()
        self._initialize_default_stakeholders()

    def _initialize_default_constraints(self):
        """Register the example's fairness, equity, and proxy constraints."""

        def fairness_check(prices, market_context):
            base_prices = np.asarray(market_context.get("base_prices", prices), dtype=float)
            if np.any(base_prices <= 0):
                raise ValueError("Reference prices must be positive.")
            return float(np.mean(prices / base_prices)) <= 1.5

        self.constraints.append(
            EthicalConstraint(
                EthicalPrinciple.FAIRNESS,
                "hard",
                "Limit the mean price increase relative to reference prices",
                fairness_check,
                penalty_weight=2.0,
                threshold=1.5,
            )
        )

        def equity_check(prices, market_context):
            segments = market_context.get("customer_segments", {})
            small_indices = segments.get("small", [])
            if not small_indices:
                return True
            small_prices = prices[small_indices]
            if len(small_prices) == 0:
                return True
            return float(np.mean(small_prices)) <= float(np.mean(prices)) * 1.2

        self.constraints.append(
            EthicalConstraint(
                EthicalPrinciple.EQUITY,
                "hard",
                "Limit the average price premium for small shipping lines",
                equity_check,
                penalty_weight=1.5,
                threshold=1.2,
            )
        )

        def non_discrimination_check(prices, market_context):
            average_price = float(np.mean(prices))
            if average_price <= 0:
                raise ValueError("Average price must be positive.")
            coefficient_of_variation = float(np.std(prices) / average_price)
            return coefficient_of_variation <= 0.3

        self.constraints.append(
            EthicalConstraint(
                EthicalPrinciple.NON_DISCRIMINATION,
                "soft",
                "Limit relative price variation as a proxy for discrimination",
                non_discrimination_check,
                penalty_weight=1.0,
                threshold=0.3,
            )
        )

        def sustainability_check(prices, market_context):
            impact = market_context.get("environmental_impact", 0.5)
            return 0.0 <= impact <= 0.8

        self.constraints.append(
            EthicalConstraint(
                EthicalPrinciple.SUSTAINABILITY,
                "soft",
                "Keep the supplied environmental-impact index within its threshold",
                sustainability_check,
                penalty_weight=0.8,
                threshold=0.8,
            )
        )

    def _initialize_default_stakeholders(self):
        """Register utility functions and relative stakeholder weights."""

        def customer_utility(prices, context):
            service_quality = context.get("service_quality", 0.8)
            sensitivity = context.get("price_sensitivity", 0.01)
            return max(0.0, service_quality - sensitivity * float(np.mean(prices)))

        self.stakeholders.append(
            Stakeholder(
                StakeholderType.CUSTOMERS,
                "Shipping Lines",
                ["affordable_pricing", "service_quality", "reliability"],
                customer_utility,
                weight=2.0,
            )
        )

        def shareholder_utility(prices, context):
            costs = np.asarray(context.get("costs", np.ones_like(prices) * 80), dtype=float)
            demands = np.asarray(context.get("demands", np.ones_like(prices) * 10000), dtype=float)
            if costs.shape != prices.shape or demands.shape != prices.shape:
                raise ValueError("Prices, costs, and demands must have matching shapes.")
            return float(np.sum((prices - costs) * demands) / 1_000_000)

        self.stakeholders.append(
            Stakeholder(
                StakeholderType.SHAREHOLDERS,
                "Port Investors",
                ["profitability", "market_share", "growth"],
                shareholder_utility,
                weight=1.5,
            )
        )

        def community_utility(prices, context):
            employment = context.get("employment_level", 0.9)
            local_business = context.get("local_business_impact", 0.8)
            price_factor = 1.0 if float(np.mean(prices)) < 120 else 0.8
            return employment * local_business * price_factor

        self.stakeholders.append(
            Stakeholder(
                StakeholderType.COMMUNITY,
                "Local Community",
                ["employment", "local_business", "economic_development"],
                community_utility,
                weight=1.2,
            )
        )

        def environment_utility(prices, context):
            return 1.0 - context.get("environmental_impact", 0.5)

        self.stakeholders.append(
            Stakeholder(
                StakeholderType.ENVIRONMENT,
                "Environment",
                ["sustainability", "pollution_reduction", "carbon_footprint"],
                environment_utility,
                weight=1.0,
            )
        )

    def evaluate_ethical_compliance(self, prices: np.ndarray, market_context: Dict) -> Dict:
        """Return per-constraint results, violations, penalties, and score."""

        prices = np.asarray(prices, dtype=float)
        if prices.ndim != 1 or prices.size == 0 or not np.all(np.isfinite(prices)):
            raise ValueError("Prices must be a non-empty, finite one-dimensional array.")

        results = {
            "overall_compliance": True,
            "hard_constraints_satisfied": True,
            "constraint_compliance": {},
            "ethical_score": 1.0,
            "violations": [],
            "penalties": [],
            "recommendations": [],
        }
        total_penalty = 0.0

        for constraint in self.constraints:
            try:
                compliant = bool(constraint.check_function(prices, market_context))
            except (ValueError, IndexError, TypeError, FloatingPointError) as error:
                compliant = False
                description = f"{constraint.description} (evaluation error: {error})"
            else:
                description = constraint.description

            principle = constraint.principle.value
            results["constraint_compliance"][principle] = compliant
            if compliant:
                continue

            results["overall_compliance"] = False
            if constraint.constraint_type == "hard":
                results["hard_constraints_satisfied"] = False
                penalty = constraint.penalty_weight
            else:
                penalty = constraint.penalty_weight * self._calculate_violation_degree(
                    prices, market_context, constraint
                )

            total_penalty += penalty
            results["penalties"].append(penalty)
            results["violations"].append(
                {
                    "principle": principle,
                    "description": description,
                    "constraint_type": constraint.constraint_type,
                }
            )
            results["recommendations"].append(
                self._generate_compliance_recommendation(constraint)
            )

        results["ethical_score"] = max(0.0, 1.0 - total_penalty)
        return results

    def _calculate_violation_degree(
        self, prices: np.ndarray, market_context: Dict, constraint: EthicalConstraint
    ) -> float:
        """Scale soft-constraint violations to the interval from zero to one."""

        if constraint.principle == EthicalPrinciple.NON_DISCRIMINATION:
            average_price = float(np.mean(prices))
            if average_price <= 0:
                return 1.0
            coefficient = float(np.std(prices) / average_price)
            degree = max(0.0, (coefficient - constraint.threshold) / constraint.threshold)
            return min(degree, 1.0)

        if constraint.principle == EthicalPrinciple.SUSTAINABILITY:
            impact = float(market_context.get("environmental_impact", 0.5))
            degree = max(0.0, (impact - constraint.threshold) / (1.0 - constraint.threshold))
            return min(degree, 1.0)

        return 0.5

    def _generate_compliance_recommendation(self, constraint: EthicalConstraint) -> str:
        """Provide a principle-specific prompt for reviewing a violation."""

        recommendations = {
            EthicalPrinciple.FAIRNESS: "Review price increases against the reference prices.",
            EthicalPrinciple.EQUITY: "Review affordability for small shipping lines.",
            EthicalPrinciple.NON_DISCRIMINATION: "Review the reasons for price differences.",
            EthicalPrinciple.SUSTAINABILITY: "Review environmental impacts and mitigation options.",
        }
        return recommendations.get(
            constraint.principle,
            f"Review the {constraint.principle.value} principle.",
        )

    def calculate_stakeholder_utilities(self, prices: np.ndarray, market_context: Dict) -> Dict[str, float]:
        """Calculate each modeled stakeholder's weighted utility."""

        prices = np.asarray(prices, dtype=float)
        utilities = {}
        for stakeholder in self.stakeholders:
            utility = stakeholder.utility_function(prices, market_context)
            utilities[stakeholder.name] = float(utility) * stakeholder.weight
        return utilities

    def optimize_ethical_pricing(self, base_prices: np.ndarray, market_context: Dict) -> EthicalDecision:
        """Use the specified 100-step decreasing-step random-walk search."""

        optimal_prices = np.asarray(base_prices, dtype=float).copy()
        if optimal_prices.ndim != 1 or optimal_prices.size == 0:
            raise ValueError("Base prices must be a non-empty one-dimensional array.")

        best_score = -np.inf
        best_utilities = None
        best_compliance = None

        for iteration in range(100):
            candidate_prices = self._generate_candidate_prices(
                optimal_prices, market_context, iteration
            )
            compliance = self.evaluate_ethical_compliance(candidate_prices, market_context)

            # Hard constraints define feasibility; soft violations affect the score instead.
            if not compliance["hard_constraints_satisfied"]:
                continue

            utilities = self.calculate_stakeholder_utilities(candidate_prices, market_context)
            overall_utility = sum(utilities.values())
            combined_score = compliance["ethical_score"] * 0.6 + (overall_utility / 10.0) * 0.4

            if combined_score > best_score:
                best_score = combined_score
                optimal_prices = candidate_prices.copy()
                best_utilities = utilities.copy()
                best_compliance = compliance.copy()

        if best_compliance is None:
            raise RuntimeError("The search found no candidate satisfying all hard constraints.")

        decision = EthicalDecision(
            timestamp=time.time(),
            decision_maker="EthicalFramework",
            prices=optimal_prices.copy(),
            ethical_score=best_compliance["ethical_score"],
            stakeholder_utilities=best_utilities,
            constraint_compliance=best_compliance["constraint_compliance"],
            rationale=self._generate_decision_rationale(best_compliance, best_utilities),
            audit_trail=self._create_audit_trail(optimal_prices, market_context),
            approval_required=best_compliance["ethical_score"] < 0.8,
        )
        self.audit_log.append(decision)
        return decision

    def _generate_candidate_prices(
        self, current_prices: np.ndarray, market_context: Dict, iteration: int
    ) -> np.ndarray:
        """Perturb each price and enforce the market's elementwise bounds."""

        step_size = 10.0 * (0.95 ** iteration)
        candidate_prices = np.asarray(current_prices, dtype=float).copy()
        perturbations = np.random.uniform(-step_size, step_size, len(candidate_prices))
        candidate_prices += perturbations
        minimum = market_context.get("min_price", 80)
        maximum = market_context.get("max_price", 150)
        if minimum > maximum:
            raise ValueError("The minimum price cannot exceed the maximum price.")
        return np.clip(candidate_prices, minimum, maximum)

    def _generate_decision_rationale(self, compliance: Dict, utilities: Dict) -> str:
        """Summarize constraint status and the largest weighted utility."""

        rationale = [f"Ethical score: {compliance['ethical_score']:.1%}"]
        if compliance["violations"]:
            rationale.append(f"Soft concerns recorded: {len(compliance['violations'])}")
        else:
            rationale.append("All modeled constraints satisfied")
        if utilities:
            leading_stakeholder = max(utilities.items(), key=lambda item: item[1])[0]
            rationale.append(f"Largest weighted utility: {leading_stakeholder}")
        return "; ".join(rationale)

    def _create_audit_trail(self, prices: np.ndarray, market_context: Dict) -> List[str]:
        """Record the decision context and price summary for later review."""

        return [
            f"Decision timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}",
            f"Average price: ${np.mean(prices):.2f}",
            f"Price range: ${np.min(prices):.2f} - ${np.max(prices):.2f}",
            f"Market conditions: {market_context.get('market_state', 'normal')}",
            "Framework version: 1.0",
        ]

    def get_compliance_report(self) -> Dict:
        """Summarize decisions recorded by this framework instance."""

        if not self.audit_log:
            return {"message": "No decisions recorded."}

        total = len(self.audit_log)
        scores = [decision.ethical_score for decision in self.audit_log]
        compliant = sum(score >= 0.8 for score in scores)
        constraint_stats = defaultdict(lambda: {"compliant": 0, "total": 0})

        for decision in self.audit_log:
            for principle, is_compliant in decision.constraint_compliance.items():
                constraint_stats[principle]["total"] += 1
                constraint_stats[principle]["compliant"] += int(is_compliant)

        principle_compliance = {
            principle: counts["compliant"] / counts["total"]
            for principle, counts in constraint_stats.items()
        }
        return {
            "total_decisions": total,
            "average_ethical_score": float(np.mean(scores)),
            "compliance_rate": compliant / total,
            "principle_compliance": principle_compliance,
            "recent_decisions": self.audit_log[-5:],
        }

In [ ]:
class BiasDetector:
    """Flag large between-group price differences as a screening signal."""

    def __init__(self):
        self.bias_indicators = []
        self.mitigation_strategies = {}

    def detect_price_bias(self, prices: np.ndarray, customer_attributes: Dict) -> Dict:
        """Assess size and regional price variation using explicit thresholds."""

        prices = np.asarray(prices, dtype=float)
        if prices.ndim != 1 or not np.all(np.isfinite(prices)):
            raise ValueError("Prices must be a finite one-dimensional array.")

        analysis = {
            "bias_detected": False,
            "bias_types": [],
            "severity": "low",
            "recommendations": [],
        }
        checks = [
            ("size_categories", "size_based", 0.2, "Review pricing across customer sizes."),
            ("regions", "regional", 0.15, "Review regional pricing differences."),
        ]

        for attribute_key, bias_name, threshold, recommendation in checks:
            if attribute_key not in customer_attributes:
                continue
            categories = customer_attributes[attribute_key]
            if len(categories) != len(prices):
                raise ValueError(f"{attribute_key} must contain one category per price.")
            difference = self._check_group_bias(prices, categories)
            if difference > threshold:
                analysis["bias_detected"] = True
                analysis["bias_types"].append(bias_name)
                analysis["severity"] = "high" if difference > 2 * threshold else "medium"
                analysis["recommendations"].append(recommendation)
        return analysis

    def _check_group_bias(self, prices: np.ndarray, categories: List[str]) -> float:
        """Return the largest relative difference between category means."""

        grouped_prices = defaultdict(list)
        for price, category in zip(prices, categories):
            grouped_prices[category].append(float(price))
        group_means = [float(np.mean(values)) for values in grouped_prices.values()]
        if len(group_means) < 2:
            return 0.0
        minimum = min(group_means)
        if minimum <= 0:
            return float("inf")
        return (max(group_means) - minimum) / minimum

    def _check_size_bias(self, prices: np.ndarray, size_categories: List[str]) -> float:
        """Compatibility helper for checking price variation by group."""

        return self._check_group_bias(np.asarray(prices, dtype=float), size_categories)

    def _check_regional_bias(self, prices: np.ndarray, regions: List[str]) -> float:
        """Compatibility helper for checking price variation by region."""

        return self._check_group_bias(np.asarray(prices, dtype=float), regions)


class TransparencyEngine:
    """Produce a human-readable summary of a recorded pricing decision."""

    def __init__(self):
        self.transparency_records = []
        self.explanation_templates = {}

    def generate_pricing_explanation(self, decision: EthicalDecision, market_context: Dict) -> Dict:
        """Summarize modeled factors without claiming complete explainability."""

        explanation = {
            "summary": f"Pricing decision with ethical score {decision.ethical_score:.1%}",
            "factors": [
                f"Market conditions: {market_context.get('market_state', 'normal')}",
                f"Average price: ${np.mean(decision.prices):.2f}",
                f"Price range: ${np.min(decision.prices):.2f} - ${np.max(decision.prices):.2f}",
            ],
            "ethical_considerations": [],
            "stakeholder_impact": [],
            "alternatives_considered": [],
            "confidence_level": "Not assessed",
        }
        for principle, compliant in decision.constraint_compliance.items():
            status = "Satisfied" if compliant else "Not satisfied"
            explanation["ethical_considerations"].append(
                f"{principle.replace('_', ' ').title()}: {status}"
            )
        for stakeholder, utility in decision.stakeholder_utilities.items():
            explanation["stakeholder_impact"].append(
                f"{stakeholder}: weighted utility {utility:.3f}"
            )
        return explanation

In [ ]:
def create_concrete_example():
    """Create the three-port, three-shipping-line example instance."""

    ports = [
        Port(id=0, name="Port A", capacity=20000, costs=[80, 85, 82]),
        Port(id=1, name="Port B", capacity=18000, costs=[85, 78, 88]),
        Port(id=2, name="Port C", capacity=22000, costs=[75, 90, 80]),
    ]
    shipping_lines = [
        ShippingLine(0, "Global Shipping Line", 12000, 1, "large", 0.008),
        ShippingLine(1, "Pacific Trade Line", 8000, 2, "medium", 0.012),
        ShippingLine(2, "Regional Container Line", 3000, 3, "small", 0.015),
    ]
    market_params = MarketParameters(price_sensitivity=0.01, min_price=80, max_price=150)
    return ports, shipping_lines, market_params


ports, shipping_lines, market_params = create_concrete_example()
ethical_framework = EthicalFramework()

print("Value-Aligned and Ethical Framework Configuration")
print(f"Number of ports: {len(ports)}")
print(f"Number of shipping lines: {len(shipping_lines)}")
print(f"Total market demand: {sum(line.demand for line in shipping_lines):,} TEU")
print(f"Total port capacity: {sum(port.capacity for port in ports):,} TEU")
print(f"Ethical principles represented: {[item.value for item in ethical_framework.ethical_principles]}")
print(f"Constraints evaluated: {len(ethical_framework.constraints)}")
print(f"Stakeholders with utility functions: {len(ethical_framework.stakeholders)}")
print("\nShipping line characteristics:")
for line in shipping_lines:
    print(
        f"- {line.name}: {line.size_category.title()}, "
        f"priority {line.priority}, price sensitivity {line.price_sensitivity:.3f}"
    )

## Inspecting a shortage scenario

The next cell evaluates a supplied high-price vector and then searches for a hard-feasible candidate. The reference prices are used by the fairness test, while the small-customer segment is identified by its index in the price vector. The profit calculation uses the supplied costs and demands; it is a simple contribution calculation, not a demand-response or capacity-constrained profit model.

The framework's overall compliance flag includes both hard and soft constraints. The search excludes candidates that fail hard constraints, while soft violations reduce the ethical score. These two concepts are therefore reported separately.

In [ ]:
def demonstrate_ethical_pricing():
    """Evaluate the illustrative shortage prices and search for a decision."""

    market_context = {
        "market_state": "shortage",
        "base_prices": np.array([100.0, 105.0, 95.0]),
        "min_price": market_params.min_price,
        "max_price": market_params.max_price,
        "customer_segments": {"small": [2], "medium": [1], "large": [0]},
        "demands": np.array([12000.0, 8000.0, 3000.0]),
        "costs": np.array([80.0, 85.0, 75.0]),
        "service_quality": 0.85,
        "price_sensitivity": 0.01,
        "employment_level": 0.92,
        "local_business_impact": 0.88,
        "environmental_impact": 0.6,
    }
    unconstrained_prices = np.array([145.0, 142.0, 138.0])

    print("\nETHICAL PRICING DECISION DEMONSTRATION")
    print("Scenario: market shortage")
    print(f"Illustrative comparison prices: ${', '.join(f'{price:.0f}' for price in unconstrained_prices)}")
    print(f"Mean comparison price: ${np.mean(unconstrained_prices):.2f}")

    comparison = ethical_framework.evaluate_ethical_compliance(
        unconstrained_prices, market_context
    )
    print("\nAssessment of comparison prices")
    print(f"All constraints satisfied: {'Yes' if comparison['overall_compliance'] else 'No'}")
    print(f"Hard constraints satisfied: {'Yes' if comparison['hard_constraints_satisfied'] else 'No'}")
    print(f"Ethical score: {comparison['ethical_score']:.1%}")
    for violation in comparison["violations"]:
        print(
            f"- {violation['principle'].replace('_', ' ').title()} "
            f"({violation['constraint_type']}): {violation['description']}"
        )

    decision = ethical_framework.optimize_ethical_pricing(
        unconstrained_prices, market_context
    )
    print("\nRandom-walk search decision")
    print(f"Selected prices: ${', '.join(f'{price:.2f}' for price in decision.prices)}")
    print(f"Mean selected price: ${np.mean(decision.prices):.2f}")
    print(f"Ethical score: {decision.ethical_score:.1%}")
    print(f"Approval required by this rule: {'Yes' if decision.approval_required else 'No'}")
    print(f"Rationale: {decision.rationale}")

    print("\nConstraint assessment")
    for principle, compliant in decision.constraint_compliance.items():
        print(f"- {principle.replace('_', ' ').title()}: {'Satisfied' if compliant else 'Not satisfied'}")

    print("\nWeighted stakeholder utilities")
    for stakeholder, utility in decision.stakeholder_utilities.items():
        print(f"- {stakeholder}: {utility:.3f}")

    unconstrained_profit = float(
        np.sum((unconstrained_prices - market_context["costs"]) * market_context["demands"])
    )
    selected_profit = float(
        np.sum((decision.prices - market_context["costs"]) * market_context["demands"])
    )
    if not np.isclose(unconstrained_profit, 0.0):
        profit_reduction = (unconstrained_profit - selected_profit) / unconstrained_profit * 100
    else:
        profit_reduction = np.nan
    print("\nContribution comparison under fixed demands")
    print(f"Comparison-price contribution: ${unconstrained_profit:,.2f}")
    print(f"Selected-price contribution: ${selected_profit:,.2f}")
    print(f"Relative change from comparison prices: {profit_reduction:.1f}%")

    explanation = ethical_framework.transparency_engine.generate_pricing_explanation(
        decision, market_context
    )
    print("\nDecision explanation")
    print(explanation["summary"])
    for factor in explanation["factors"]:
        print(f"- {factor}")
    for item in explanation["ethical_considerations"]:
        print(f"- {item}")
    return decision, comparison, market_context


ethical_decision, unconstrained_compliance, shortage_context = demonstrate_ethical_pricing()

## Bias screening and mitigation illustration

A group-price difference is a screening signal, not proof of discriminatory intent: valid cost, service, or contractual factors may explain differences. The detector compares group means and applies the stated relative-difference thresholds. The mitigation demonstration repeatedly shrinks deviations from the overall mean and checks the detector again. This mechanical transformation illustrates the feedback loop; it does not determine whether equal or more similar prices are appropriate in practice.

In [ ]:
def demonstrate_bias_detection():
    """Detect group-price differences and demonstrate a simple reduction step."""

    prices = np.array([120.0, 125.0, 140.0])
    customer_attributes = {
        "size_categories": ["large", "medium", "small"],
        "regions": ["europe", "asia", "africa"],
    }
    detector = BiasDetector()
    initial_analysis = detector.detect_price_bias(prices, customer_attributes)

    print("\nBIAS DETECTION AND MITIGATION")
    print(f"Test prices: ${', '.join(f'{price:.0f}' for price in prices)}")
    print(f"Customer attributes: {customer_attributes}")
    print(f"Bias signal detected: {'Yes' if initial_analysis['bias_detected'] else 'No'}")
    print(f"Severity: {initial_analysis['severity']}")
    print(f"Flagged comparisons: {initial_analysis['bias_types'] or 'None'}")
    for recommendation in initial_analysis["recommendations"]:
        print(f"- {recommendation}")

    mitigated_prices = prices.copy()
    for _ in range(100):
        analysis = detector.detect_price_bias(mitigated_prices, customer_attributes)
        if not analysis["bias_detected"]:
            break
        mean_price = float(np.mean(mitigated_prices))
        mitigated_prices = mean_price + 0.9 * (mitigated_prices - mean_price)

    final_analysis = detector.detect_price_bias(mitigated_prices, customer_attributes)
    print("\nIllustrative reduction of price differences")
    print(f"Starting prices: ${', '.join(f'{price:.2f}' for price in prices)}")
    print(f"Resulting prices: ${', '.join(f'{price:.2f}' for price in mitigated_prices)}")
    print(f"Bias signal remains: {'Yes' if final_analysis['bias_detected'] else 'No'}")
    return mitigated_prices, initial_analysis, final_analysis


mitigated_prices, initial_bias_analysis, final_bias_analysis = demonstrate_bias_detection()

## Repeated decisions under varying market conditions

The simulation samples market states and context values, then calls the same ethical-pricing search for each decision. The random sampling is intentionally unseeded, so the resulting statistics and plots are run-specific. The compliance rate below is defined as the share of decisions with ethical score at least 0.8; it is not a legal compliance rate. A separate framework instance keeps the simulation report limited to simulation decisions rather than mixing in the earlier demonstration.

In [ ]:
def run_ethical_simulation(num_decisions: int = 30):
    """Run repeated random-context decisions using the same search method."""

    if num_decisions <= 0:
        raise ValueError("num_decisions must be positive.")

    simulation_framework = EthicalFramework()
    results = {
        "decisions": [],
        "ethical_scores": [],
        "compliance_rates": [],
        "profit_impacts": [],
        "stakeholder_utilities": [],
        "bias_detections": [],
        "market_states": [],
        "framework": simulation_framework,
    }
    market_states = ["normal", "shortage", "surplus", "disruption"]

    print("\nCOMPREHENSIVE ETHICAL FRAMEWORK SIMULATION")
    print(f"Number of decisions: {num_decisions}")

    for decision_number in range(num_decisions):
        market_state = random.choice(market_states)
        demand_factor = 1.0
        if market_state == "shortage":
            demand_factor = 1.4
        elif market_state == "surplus":
            demand_factor = 0.8
        elif market_state == "disruption":
            demand_factor = random.uniform(0.6, 1.3)

        context = {
            "market_state": market_state,
            "base_prices": np.random.uniform(90.0, 110.0, 3),
            "min_price": market_params.min_price,
            "max_price": market_params.max_price,
            "customer_segments": {"small": [2], "medium": [1], "large": [0]},
            "demands": np.array([12000.0, 8000.0, 3000.0]) * demand_factor,
            "costs": np.array([80.0, 85.0, 75.0]),
            "service_quality": random.uniform(0.7, 0.95),
            "price_sensitivity": 0.01,
            "employment_level": random.uniform(0.85, 0.95),
            "local_business_impact": random.uniform(0.8, 0.9),
            "environmental_impact": random.uniform(0.4, 0.8),
        }
        base_prices = context["base_prices"].copy()
        if market_state == "shortage":
            base_prices *= 1.3
        elif market_state == "surplus":
            base_prices *= 0.9

        decision = simulation_framework.optimize_ethical_pricing(base_prices, context)
        score = decision.ethical_score
        compliance_rate = float(score >= 0.8)
        profit = float(np.sum((decision.prices - context["costs"]) * context["demands"]))
        total_utility = float(sum(decision.stakeholder_utilities.values()))
        bias = BiasDetector().detect_price_bias(
            decision.prices,
            {"size_categories": ["large", "medium", "small"]},
        )

        results["decisions"].append(decision)
        results["ethical_scores"].append(score)
        results["compliance_rates"].append(compliance_rate)
        results["profit_impacts"].append(profit)
        results["stakeholder_utilities"].append(total_utility)
        results["bias_detections"].append(int(bias["bias_detected"]))
        results["market_states"].append(market_state)

        if decision_number % 10 == 0 or decision_number == num_decisions - 1:
            average_score = float(np.mean(results["ethical_scores"]))
            average_compliance = float(np.mean(results["compliance_rates"]))
            cumulative_profit = float(sum(results["profit_impacts"]))
            print(
                f"Decision {decision_number}: mean ethical score={average_score:.1%}, "
                f"score-threshold rate={average_compliance:.1%}, "
                f"cumulative contribution=${cumulative_profit:,.0f}"
            )

    return results


simulation_results = run_ethical_simulation(30)
compliance_report = simulation_results["framework"].get_compliance_report()
avg_ethical_score = float(np.mean(simulation_results["ethical_scores"]))
avg_compliance = float(np.mean(simulation_results["compliance_rates"]))
total_profit = float(sum(simulation_results["profit_impacts"]))
avg_utility = float(np.mean(simulation_results["stakeholder_utilities"]))
bias_incidence_rate = float(np.mean(simulation_results["bias_detections"]))

print("\nSIMULATION COMPLIANCE REPORT")
print(f"Decisions recorded: {compliance_report['total_decisions']}")
print(f"Mean ethical score: {compliance_report['average_ethical_score']:.1%}")
print(f"Score-threshold rate: {compliance_report['compliance_rate']:.1%}")
print("Constraint satisfaction rates:")
for principle, rate in compliance_report["principle_compliance"].items():
    print(f"- {principle.replace('_', ' ').title()}: {rate:.1%}")

print("\nSimulation summary")
print(f"Mean ethical score: {avg_ethical_score:.1%}")
print(f"Score-threshold rate: {avg_compliance:.1%}")
print(f"Total contribution under fixed-demand calculation: ${total_profit:,.2f}")
print(f"Mean total weighted stakeholder utility: {avg_utility:.3f}")
print(f"Bias-screening incidence: {bias_incidence_rate:.1%}")

## Visualizing simulation outcomes

Each panel summarizes a different modeled quantity. The ethical-score line is shown by decision; the horizontal line marks the 0.8 approval threshold used in the code. The bias panel uses a five-decision rolling window, with shorter windows at the beginning. These plots describe this simulation run only and should not be interpreted as evidence about an operating port.

In [ ]:
def visualize_ethical_framework():
    """Plot simulation scores, constraint rates, utilities, profit, and bias flags."""

    figure, axes = plt.subplots(2, 3, figsize=(18, 12))
    figure.suptitle("Value-Aligned and Ethical Framework: Simulation Summary", fontsize=16)

    steps = np.arange(len(simulation_results["ethical_scores"]))
    axes[0, 0].plot(steps, simulation_results["ethical_scores"], linewidth=2)
    axes[0, 0].axhline(0.8, color="red", linestyle="--", label="Approval threshold (0.8)")
    axes[0, 0].set(title="Ethical score by decision", xlabel="Decision number", ylabel="Score")
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    principles = list(compliance_report["principle_compliance"])
    rates = list(compliance_report["principle_compliance"].values())
    axes[0, 1].bar([name.replace("_", " ").title() for name in principles], rates)
    axes[0, 1].set(title="Constraint satisfaction by principle", ylabel="Rate")
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].tick_params(axis="x", rotation=35)
    axes[0, 1].grid(True, axis="y", alpha=0.3)

    stakeholder_names = list(simulation_results["decisions"][0].stakeholder_utilities)
    mean_utilities = [
        np.mean([decision.stakeholder_utilities[name] for decision in simulation_results["decisions"]])
        for name in stakeholder_names
    ]
    axes[0, 2].bar(stakeholder_names, mean_utilities, color="green", alpha=0.7)
    axes[0, 2].set(title="Mean weighted stakeholder utility", ylabel="Utility")
    axes[0, 2].tick_params(axis="x", rotation=30)
    axes[0, 2].grid(True, axis="y", alpha=0.3)

    axes[1, 0].scatter(
        simulation_results["ethical_scores"],
        simulation_results["profit_impacts"],
        alpha=0.7,
    )
    axes[1, 0].set(
        title="Contribution and ethical score",
        xlabel="Ethical score",
        ylabel="Contribution under fixed demands ($)",
    )
    axes[1, 0].grid(True, alpha=0.3)

    flags = simulation_results["bias_detections"]
    window_size = 5
    rolling_rates = [
        float(np.mean(flags[max(0, index - window_size + 1):index + 1]))
        for index in range(len(flags))
    ]
    axes[1, 1].plot(steps, rolling_rates, color="purple", linewidth=2)
    axes[1, 1].set(
        title="Bias-screening incidence (rolling window of up to 5)",
        xlabel="Decision number",
        ylabel="Incidence rate",
        ylim=(0, 1),
    )
    axes[1, 1].grid(True, alpha=0.3)

    summary_names = ["Mean ethical score", "Score-threshold rate", "No bias flag"]
    summary_values = [avg_ethical_score, avg_compliance, 1.0 - bias_incidence_rate]
    axes[1, 2].bar(summary_names, summary_values, color=["blue", "green", "orange"], alpha=0.7)
    axes[1, 2].set(title="Run summary", ylabel="Rate or score", ylim=(0, 1))
    axes[1, 2].tick_params(axis="x", rotation=25)
    axes[1, 2].grid(True, axis="y", alpha=0.3)

    for axis in axes.flat:
        axis.margins(x=0.08)
    figure.tight_layout()
    plt.show()
    return figure


visualization_fig = visualize_ethical_framework()

## Reporting the observed run

The table below contains metrics calculated from the current simulation, not benchmark values for other pricing methods. Cross-method comparisons require results produced on the same data, under aligned assumptions, and with clearly defined metrics. Those comparison data are not part of this example, so no relative performance claim is made here.

In [ ]:
def report_simulation_results():
    """Present only metrics calculated from the simulation in this run."""

    report = pd.DataFrame(
        [
            {"Metric": "Number of decisions", "Observed value": len(simulation_results["decisions"])},
            {"Metric": "Mean ethical score", "Observed value": avg_ethical_score},
            {"Metric": "Score-threshold rate (score >= 0.8)", "Observed value": avg_compliance},
            {"Metric": "Total fixed-demand contribution ($)", "Observed value": total_profit},
            {"Metric": "Mean total weighted stakeholder utility", "Observed value": avg_utility},
            {"Metric": "Bias-screening incidence", "Observed value": bias_incidence_rate},
        ]
    )
    print("Observed simulation metrics")
    display(report)
    return report


simulation_report = report_simulation_results()

## Interpretation and limitations

The framework makes stakeholder weights, constraint thresholds, and candidate scoring visible so that a reader can inspect how they influence the selected prices. Hard constraints determine search feasibility; soft constraints contribute to the ethical-score penalty. The stochastic search explores a limited sequence of nearby prices and returns the best eligible candidate it encounters. A different random sequence can lead to a different decision, and the method does not certify global optimality.

The reported score is not a percentage of real-world ethical compliance. It is a bounded score derived from the penalties defined in this notebook. Likewise, the bias detector measures price differences across supplied groups but does not test causal discrimination or account for legitimate cost and service differences. The profit calculation holds demand fixed and does not enforce port capacities or model customer substitution. Consequently, the numerical outputs are illustrative results of this model, not evidence that a real pricing system is lawful, fair, transparent, or commercially optimal.

A practical deployment would require validated data, stakeholder and legal review, documented justifications for group-level price differences, and monitoring procedures appropriate to the operating context. The natural extensions are to refine the input data and ethical criteria while retaining the distinction between hard feasibility checks, soft penalties, stakeholder utility, and audit reporting.